In [1]:
"""
download_govuk_strategy.py
---------------------------
Downloads UK net-zero / carbon strategy and delivery-plan PDFs from
GOV.UK using the official public Search API + Content API.

Run as a plain Python script (not a notebook):
    pip install requests
    python download_govuk_strategy.py

Or paste the entire file into a single notebook cell.

FIXES vs previous version:
  - All functions defined before main() so notebook execution works
  - count bumped to 1500 (API max) + pagination so no results are missed
  - End date corrected to 2026-06-30 (June has 30 days, not 31)
  - extract_pdf_urls has a fallback regex scan for non-standard pages
"""

import csv
import re
import time
from pathlib import Path
from urllib.parse import urlencode

import requests

# -----------------------------------------------------------------------
# CONFIGURATION
# -----------------------------------------------------------------------

OUTPUT_DIR = Path("govuk_raw")
METADATA_FILE = Path("govuk_metadata.csv")

SEARCH_URL = "https://www.gov.uk/api/search.json"
CONTENT_API_BASE = "https://www.gov.uk/api/content"
SITE_BASE = "https://www.gov.uk"

REQUEST_TIMEOUT = 30
RETRY_ATTEMPTS = 3
RETRY_BACKOFF = 5
POLITE_DELAY = 1.0

HEADERS = {
    "User-Agent": "NTU-MSc-Research-RCCB-Index/1.0 (academic research)"
}

START_DATE = "2008-01-01"
END_DATE   = "2026-06-30"   # June has 30 days -- was incorrectly 31

# -----------------------------------------------------------------------
# INCLUSION RULE
# Including all three predecessor departments so no documents are missed
# due to machinery-of-government changes:
#   DECC  2008-2016, BEIS 2016-2023, DESNZ 2023-present
# -----------------------------------------------------------------------

ORGANISATIONS = [
    "department-for-energy-security-and-net-zero",
    "department-for-business-energy-and-industrial-strategy",
    "department-of-energy-climate-change",
]

KEYWORDS = [
    "net zero strategy",
    "carbon budget delivery plan",
    "powering up britain",
    "ten point plan green industrial revolution",
    "nationally determined contribution",
    "decarbonisation strategy",
]


# -----------------------------------------------------------------------
# STEP 1 HELPERS -- search the GOV.UK Search API
# -----------------------------------------------------------------------

def _get_json(url):
    """GET a URL with retries, return parsed JSON or None."""
    for attempt in range(1, RETRY_ATTEMPTS + 1):
        try:
            resp = requests.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
            if resp.status_code == 200:
                return resp.json()
            print("    [WARN] HTTP {} (attempt {}/{})".format(
                resp.status_code, attempt, RETRY_ATTEMPTS))
        except requests.RequestException as exc:
            print("    [ERROR] {} (attempt {}/{})".format(exc, attempt, RETRY_ATTEMPTS))
        except ValueError as exc:
            print("    [ERROR] JSON parse failed: {}".format(exc))
            return None
        if attempt < RETRY_ATTEMPTS:
            time.sleep(RETRY_BACKOFF * attempt)
    return None


def search_govuk(keyword, start=0):
    """
    One page of GOV.UK search results for a keyword, starting at
    offset `start`. Returns the full JSON response or None.
    Uses count=1500 (API max) to minimise pages needed.
    """
    params = [
        ("q", keyword),
        ("count", 1500),
        ("start", start),
        ("fields", "title"),
        ("fields", "link"),
        ("fields", "description"),
        ("fields", "public_timestamp"),
        ("fields", "organisations"),
        ("filter_public_timestamp",
         "from:{},to:{}".format(START_DATE, END_DATE)),
        ("order", "-public_timestamp"),
    ]
    for org in ORGANISATIONS:
        params.append(("filter_any_organisations", org))

    url = "{}?{}".format(SEARCH_URL, urlencode(params))
    return _get_json(url)


def discover_documents():
    """
    Run every keyword query (with pagination) and return a
    de-duplicated dict keyed by GOV.UK page link.
    """
    all_docs = {}

    for keyword in KEYWORDS:
        print("Searching GOV.UK for '{}'...".format(keyword))
        start = 0

        while True:
            data = search_govuk(keyword, start=start)
            time.sleep(POLITE_DELAY)

            if not data or "results" not in data:
                print("    [WARN] no results returned")
                break

            results = data.get("results", [])
            total   = data.get("total", 0)

            for r in results:
                link = r.get("link")
                if not link:
                    continue
                if link not in all_docs:
                    orgs = r.get("organisations", [])
                    all_docs[link] = {
                        "doc_id": re.sub(r"[^a-zA-Z0-9_-]", "_",
                                         link.strip("/")),
                        "title":            r.get("title", "UNKNOWN"),
                        "link":             link,
                        "description":      r.get("description", ""),
                        "public_timestamp":  r.get("public_timestamp", ""),
                        "organisations":     ", ".join(
                            o.get("title", "") for o in orgs),
                        "matched_query":    keyword,
                    }
                else:
                    existing = all_docs[link]["matched_query"]
                    if keyword not in existing:
                        all_docs[link]["matched_query"] = \
                            "{};{}".format(existing, keyword)

            fetched_so_far = start + len(results)
            print("    fetched {}/{} (total unique: {})".format(
                fetched_so_far, total, len(all_docs)))

            if fetched_so_far >= total or not results:
                break

            start = fetched_so_far

    return list(all_docs.values())


# -----------------------------------------------------------------------
# STEP 2 HELPERS -- resolve GOV.UK page → PDF URL
# -----------------------------------------------------------------------

def fetch_content_item(link):
    """Fetch the GOV.UK Content API record for a page path."""
    url = "{}{}".format(CONTENT_API_BASE, link)
    return _get_json(url)


def extract_pdf_urls(content_item):
    """
    Extract PDF attachment URLs from a Content API record.
    Primary: details.attachments[].url (standard for policy_paper)
    Fallback: regex scan of the full JSON string (catches non-standard
              page types that nest attachments elsewhere).
    """
    pdf_urls = []
    if not content_item:
        return pdf_urls

    for att in content_item.get("details", {}).get("attachments", []):
        url = att.get("url", "")
        if url.lower().endswith(".pdf"):
            if url.startswith("/"):
                url = SITE_BASE + url
            pdf_urls.append(url)

    if not pdf_urls:
        # Fallback: scan raw JSON text for any absolute .pdf URL
        found = re.findall(r"https?://[^\s\"'\\]+\.pdf", str(content_item))
        pdf_urls.extend(dict.fromkeys(found))

    return pdf_urls


# -----------------------------------------------------------------------
# STEP 3 HELPER -- download one PDF
# -----------------------------------------------------------------------

def download_pdf(doc, pdf_url, index):
    """Download a single PDF. Returns a result dict."""
    suffix   = "" if index == 0 else "_{}".format(index + 1)
    out_file = OUTPUT_DIR / "{}{}.pdf".format(doc["doc_id"], suffix)

    for attempt in range(1, RETRY_ATTEMPTS + 1):
        try:
            resp = requests.get(pdf_url, headers=HEADERS,
                                timeout=REQUEST_TIMEOUT)
            if resp.status_code == 200 and resp.content:
                out_file.write_bytes(resp.content)
                print("  [OK] {} ({:,} bytes)".format(
                    out_file.name, len(resp.content)))
                return {"status": "success",
                        "http_code": resp.status_code,
                        "file": str(out_file),
                        "pdf_url": pdf_url}
            print("  [WARN] HTTP {} (attempt {}/{})".format(
                resp.status_code, attempt, RETRY_ATTEMPTS))
        except requests.RequestException as exc:
            print("  [ERROR] {} (attempt {}/{})".format(
                exc, attempt, RETRY_ATTEMPTS))
        if attempt < RETRY_ATTEMPTS:
            time.sleep(RETRY_BACKOFF * attempt)

    return {"status": "FAILED", "http_code": "", "file": "",
            "pdf_url": pdf_url}


# -----------------------------------------------------------------------
# MAIN
# -----------------------------------------------------------------------

def main():
    OUTPUT_DIR.mkdir(exist_ok=True)

    print("=" * 70)
    print("STEP 1: Discovering GOV.UK pages matching the inclusion rule")
    print("  Organisations: {}".format(ORGANISATIONS))
    print("  Date range: {} to {}".format(START_DATE, END_DATE))
    print("  Keywords: {}".format(KEYWORDS))
    print("=" * 70)
    documents = discover_documents()
    print("\nDiscovered {} unique candidate pages.\n".format(len(documents)))

    print("=" * 70)
    print("STEP 2: Resolving each page to its PDF attachment(s) and downloading")
    print("=" * 70)

    results = []
    for doc in documents:
        print("Resolving: {} ({})".format(doc["title"], doc["link"]))
        content_item = fetch_content_item(doc["link"])
        time.sleep(POLITE_DELAY)
        pdf_urls = extract_pdf_urls(content_item)

        if not pdf_urls:
            print("  [WARN] no PDF attachment found -- may need manual check")
            row = dict(doc)
            row.update(status="NO_PDF_FOUND", http_code="",
                       file="", pdf_url="")
            results.append(row)
            continue

        for i, pdf_url in enumerate(pdf_urls):
            dl = download_pdf(doc, pdf_url, i)
            time.sleep(POLITE_DELAY)
            row = dict(doc)
            row.update(dl)
            results.append(row)

    fieldnames = ["doc_id", "title", "link", "description",
                  "public_timestamp", "organisations", "matched_query",
                  "pdf_url", "status", "http_code", "file"]
    with open(METADATA_FILE, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(results)

    succeeded = sum(1 for r in results if r["status"] == "success")
    no_pdf    = sum(1 for r in results if r["status"] == "NO_PDF_FOUND")
    failed    = len(results) - succeeded - no_pdf

    print("\nDone. {} downloaded, {} no PDF found, {} failed ({} rows total).".format(
        succeeded, no_pdf, failed, len(results)))
    print("Audit trail: {}".format(METADATA_FILE))
    print("\nNext steps:")
    print("  1. Open {} and review titles for false positives".format(
        METADATA_FILE))
    print("  2. Manually download any NO_PDF_FOUND rows that are in scope")
    print("  3. Record your inclusion/exclusion decisions for your methodology chapter")


if __name__ == "__main__":
    main()

STEP 1: Discovering GOV.UK pages matching the inclusion rule
  Organisations: ['department-for-energy-security-and-net-zero', 'department-for-business-energy-and-industrial-strategy', 'department-of-energy-climate-change']
  Date range: 2008-01-01 to 2026-06-30
  Keywords: ['net zero strategy', 'carbon budget delivery plan', 'powering up britain', 'ten point plan green industrial revolution', 'nationally determined contribution', 'decarbonisation strategy']
Searching GOV.UK for 'net zero strategy'...
    fetched 1500/4592 (total unique: 1500)
    fetched 3000/4592 (total unique: 3000)
    fetched 4500/4592 (total unique: 4500)
    fetched 4592/4592 (total unique: 4592)
Searching GOV.UK for 'carbon budget delivery plan'...
    fetched 1500/5966 (total unique: 5211)
    fetched 3000/5966 (total unique: 5830)
    fetched 4500/5966 (total unique: 6623)
    fetched 5966/5966 (total unique: 7689)
Searching GOV.UK for 'powering up britain'...
    fetched 1500/6088 (total unique: 7919)
    fet

FileNotFoundError: [Errno 2] No such file or directory: 'govuk_raw\\government_publications_availability-of-technologies-for-provisioning-home-area-network-han-connectivity-to-electricity-and-gas-metering-equipment-communications-hub-and-in-home-devices-in-cases-where-a-2-4ghz-zigbee-wireless-han-will-not-work-effectively.pdf'